# 03 — Price Index & Inflation Measurement

## tl;dr

Phase 3 constructs matched-item Jevons and Törnqvist scanner-data cereal price indices plus a fixed-base Laspeyres-style robustness benchmark. Phase 3B independently validates Törnqvist and adds bilateral Fisher, weight/promotion/quantity, price-bounce, and extreme-relative diagnostics; Phase 3C adds bounded rolling GEKS–Törnqvist robustness. The 4,508,667 eligible comparisons span 361 adjacent-week transitions in five disconnected chain segments. This notebook focuses on price-index measurement and diagnostics; rigidity and econometric results are documented separately.

## Context & Methods

The notebook orchestrates `../src/price_indices.py`; index logic lives in reusable code. `PRICE / QTY` is the effective unit price and `(PRICE * MOVE) / QTY` is revenue, consistent with the official definitions of bundle price, bundle quantity, and item movement recorded in the Phase 1 audit.

### Key Assumptions

- Every price relative compares the same UPC/store in adjacent calendar weeks.
- Törnqvist weights use matched revenue in both transition weeks.
- Blank `SALE` is unflagged, not confirmed non-promotion; G/L remain unknown.
- The promotion sensitivity excludes B/C/S in either transition week.
- Chain links reset to 100 after missing or analytically unavailable weeks.
- Weeks 1–8 form the pre-specified multi-week Laspeyres base window.
- Extreme relatives are retained and reported without trimming or winsorization.

## Data

The validated Phase 2.1 SQLite views and versioned product-exclusion config are the only analytical inputs. The following cell intentionally rebuilds every Phase 3 output and figure.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
from pprint import pprint
from IPython.display import Image, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
INDICES = ROOT / 'data' / 'processed' / 'indices'
FIGURES = ROOT / 'report' / 'figures'
assert (ROOT / 'src' / 'price_indices.py').exists()
ROOT

In [ ]:
environment = os.environ.copy()
environment.update({
    'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1',
    'VECLIB_MAXIMUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1', 'KMP_USE_SHM': '0',
    'MPLCONFIGDIR': str(ROOT / 'work' / 'mplconfig'),
    'XDG_CACHE_HOME': str(ROOT / 'work' / 'cache'),
})
completed = subprocess.run(
    [os.sys.executable, str(ROOT / 'src' / 'price_indices.py'), '--root', str(ROOT), '--overwrite'],
    check=True, text=True, capture_output=True, env=environment,
)
print('\n'.join(completed.stdout.strip().splitlines()[-16:]))

## Results

The next cells expose bounded, machine-readable results and validation evidence. Full weekly tables remain in `data/processed/indices/`.

In [ ]:
manifest = json.loads((INDICES / 'phase3_manifest.json').read_text())
pprint({
    'sample_counts': manifest['sample_counts'],
    'matched_transition_count': manifest['matched_transition_count'],
    'chain_segments': manifest['chain_segments'],
    'extreme_relative_count': manifest['price_relative_diagnostics']['extreme_count'],
    'weak_coverage_weeks': manifest['weak_coverage_weeks'],
    'gate_decision': manifest['gate_decision'],
})

In [ ]:
with (INDICES / 'index_comparison_summary.csv').open(newline='') as handle:
    comparison_rows = list(csv.DictReader(handle))
headline = [row for row in comparison_rows if row['record_type'] == 'pair_comparison']
pprint(headline)

In [ ]:
assert manifest['sample_counts']['CORE_CEREAL_PRICE_SAMPLE'] == 4_706_611
assert manifest['matched_transition_count'] == 4_508_667
assert manifest['transition_week_count'] == 361
assert len(manifest['chain_segments']) == 5
assert all(manifest['validation'].values())
assert manifest['gate_decision'] == 'PASS WITH CONDITIONS'
manifest['validation']

## Figures

Static figures are produced by the reusable pipeline from the machine-readable weekly outputs.

In [ ]:
for figure in sorted(FIGURES.glob('0*.png')):
    display(Image(filename=str(figure), width=900))

## Takeaways

- Weighted and unweighted measurement differ materially in several transitions; this is a descriptive measurement result, not a causal finding.
- Excluding documented promotions reduces weekly volatility, but the resulting series is not a regular-price or confirmed non-promotion index.
- Decimal and hexadecimal price representations are numerically indistinguishable at reported precision.
- The three approved non-food exclusions have negligible measured index impact, but remain governed by the versioned config.
- The fixed-base benchmark loses basket coverage over time and is retained only as robustness evidence.
- **Scope:** this notebook covers price-index measurement and diagnostics; price-rigidity and econometric results are documented separately.

## Phase 3B — Chain Drift Diagnostic

This extension independently reconstructs Törnqvist from cleaned cells, constructs bilateral Fisher on the identical matched universe, and audits contemporaneous expenditure weights, promotion classifications, quantity spikes, exact/approximate price returns, and all retained extreme relatives. Existing Phase 3 outputs remain unchanged.

In [ ]:
assert (ROOT / 'src' / 'price_index_diagnostics.py').exists()
diagnostic_run = subprocess.run(
    [os.sys.executable, str(ROOT / 'src' / 'price_index_diagnostics.py'), '--root', str(ROOT), '--overwrite'],
    check=True, text=True, capture_output=True, env=environment,
)
print('\n'.join(diagnostic_run.stdout.strip().splitlines()[-14:]))

### Phase 3B results and gates

In [ ]:
import pandas as pd
diagnostic_manifest = json.loads((INDICES / 'phase3b_manifest.json').read_text())
weekly_diagnostic = pd.read_csv(INDICES / 'chain_drift_weekly_diagnostics.csv')
fisher_diagnostic = pd.read_csv(INDICES / 'fisher_tornqvist_comparison.csv')
promotion_diagnostic = pd.read_csv(INDICES / 'promotion_weight_diagnostics.csv')
bounce_diagnostic = pd.read_csv(INDICES / 'price_bounce_diagnostics.csv')
extreme_diagnostic = pd.read_csv(INDICES / 'extreme_relative_diagnostics.csv')
segment_endpoints = fisher_diagnostic.groupby('segment_id').agg(
    end_week=('week', 'max'), fisher_end=('fisher_chain_index', 'last'),
    tornqvist_end=('tornqvist_chain_index', 'last'), jevons_end=('jevons_chain_index', 'last'),
)
pprint({
    'maximum_tornqvist_factor_difference': diagnostic_manifest['maximum_independent_phase3_tornqvist_factor_difference'],
    'maximum_selected_share_sum_error': diagnostic_manifest['maximum_selected_share_sum_error'],
    'fisher_tornqvist_weekly_correlation': fisher_diagnostic['fisher_weekly_change_pct'].corr(fisher_diagnostic['tornqvist_weekly_change_pct']),
    'weeks_with_negative_current_weight_correlation_share': (weekly_diagnostic['corr_log_relative_current_revenue_share'] < 0).mean(),
    'price_bounce_episode_count': len(bounce_diagnostic),
    'extreme_relative_count': int(extreme_diagnostic.loc[extreme_diagnostic['dimension'].eq('overall'), 'extreme_count'].iloc[0]),
})
segment_endpoints

In [ ]:
assert diagnostic_manifest['matched_transition_count'] == 4_508_667
assert diagnostic_manifest['transition_week_count'] == 361
assert diagnostic_manifest['maximum_independent_phase3_tornqvist_factor_difference'] <= 1e-12
assert diagnostic_manifest['maximum_selected_share_sum_error'] <= 1e-12
assert len(diagnostic_manifest['selected_validation_weeks']) >= 15
assert diagnostic_manifest['price_bounce_episode_count'] == len(bounce_diagnostic)
assert diagnostic_manifest['extreme_relative_count'] == 29_128
assert all(diagnostic_manifest['validation'].values())
diagnostic_manifest['validation']

In [ ]:
for figure_name in ['06_fisher_tornqvist_comparison.png', '07_weight_price_interaction.png']:
    display(Image(filename=str(FIGURES / figure_name), width=900))

### Phase 3B takeaways

- Independent arithmetic confirms that the Phase 3 Törnqvist implementation is correct.
- Fisher corroborates the unusually negative weighted-index pattern.
- Negative price changes are disproportionately represented among high-current-expenditure observations; this is descriptive, not causal.
- Promotion/quantity-spike and exact price-bounce evidence is consistent with chain drift.
- A bounded rolling GEKS–Törnqvist robustness series was added in the Phase 3C section below to address this diagnostic evidence.


## Phase 3C — Rolling GEKS–Törnqvist Robustness

The approved primary method is a rolling 13-calendar-week GEKS–Törnqvist with mean splice on published. The 26-week version is sensitivity only. Windows never cross globally missing weeks, and the validated bilateral cache is reused without rewriting it. Rebuild from the repository root with `python src/geks_tornqvist.py --overwrite --reuse-bilateral`; the notebook validates and presents the resulting immutable artifacts.

In [ ]:
assert (ROOT / 'src' / 'geks_tornqvist.py').exists()
required_phase3c_outputs = [
    'geks_13_weekly.csv', 'geks_13_excl_flagged_promo.csv',
    'geks_26_sensitivity.csv', 'geks_window_diagnostics.csv',
    'geks_splice_diagnostics.csv', 'phase3c_comparison_summary.csv',
    'phase3c_manifest.json',
]
assert all((INDICES / name).exists() for name in required_phase3c_outputs)
print('Validated presence of Phase 3C artifacts; rebuild command is documented above.')

In [ ]:
phase3c_manifest = json.loads((INDICES / 'phase3c_manifest.json').read_text())
phase3c_summary = pd.read_csv(INDICES / 'phase3c_comparison_summary.csv')
geks_windows = pd.read_csv(INDICES / 'geks_window_diagnostics.csv')
primary_results = phase3c_summary.loc[
    (phase3c_summary['record_type'] == 'series_segment')
    & (phase3c_summary['series'] == 'geks_13'),
    ['segment_id', 'start_week', 'end_week', 'start_index', 'end_index',
     'segment_cumulative_change_pct', 'mean_weekly_change_pct',
     'weekly_change_volatility_sd_pp'],
]
assert all(phase3c_manifest['validation'].values())
assert phase3c_manifest['support']['13_week_segments'] == [1, 2, 3, 4, 5]
assert phase3c_manifest['support']['26_week_segments'] == [1, 2, 4, 5]
assert phase3c_manifest['final_methodology_decision'] == 'B'
assert phase3c_manifest['phase3_closed'] is True
primary_results

In [ ]:
pprint({
    'bilateral_pair_rows': phase3c_manifest['bilateral_pair_rows'],
    'estimated_window_rows': phase3c_manifest['estimated_window_rows'],
    'weak_window_review_by_specification': phase3c_manifest['weak_window_review_by_specification'],
    'validation_metrics': phase3c_manifest['validation_metrics'],
    'bounce_summary': phase3c_manifest['bounce_summary'],
    'decision': phase3c_manifest['final_methodology_decision'],
})

In [ ]:
for figure_name in [
    '06_geks_vs_bilateral_indices.png',
    '07_geks_promotion_sensitivity.png',
    '08_geks_window_coverage.png',
    '09_geks_13_vs_26.png',
]:
    display(Image(filename=str(FIGURES / figure_name), width=900))

### Phase 3 final gate

- Every required GEKS and mean-splice validation passes.
- All five contiguous segments support 13 weeks; segments 1, 2, 4, and 5 support 26 weeks.
- GEKS materially mitigates the bilateral chains' long-run downward accumulation in the long affected segments; segment 3 remains disclosed contrary evidence.
- Promotion exclusion is sensitivity only because overlap becomes weak in many windows.
- **Architecture B:** 13-week GEKS–Törnqvist is the primary expenditure-weighted cereal index; Jevons is the unweighted benchmark; bilateral Törnqvist/Fisher are chain-drift diagnostics.
- Phase 3 validation is complete; subsequent analyses are documented in later notebooks.